# SWAT output preparation for the Mujib Basin Digital Twin

This notebook prepares the SWAT hydrological outputs for use in the scenario engine
(notebook 06) and the digital twin framework. It corresponds to Section 3.3.1 of the thesis.

The SWAT model was calibrated and run by ICARDA using ArcSWAT (2024 model run).
This notebook does not calibrate or validate the SWAT model. It parses, cleans, and
restructures the raw SWAT text outputs into a consistent, analysis-ready format.

**Processing steps:**

1. Parse the raw `output.sub` text file (handles ArcSWAT formatting quirks)
2. Separate the year and area values that are glued together in the raw file
3. Standardise sub-basin identifiers across all 71 SWAT sub-basins
4. Filter to the 1988-2020 simulation period (excluding the 5-year warm-up)
5. Run quality control checks (identifier consistency, duplicate detection, completeness)
6. Export the prepared dataset as parquet and CSV

**Inputs:** Raw SWAT output files from `TxtInOut/` (output.sub, output.rch, output.hru)

**Outputs:** `swat-prepared/output_sub_FULL_FIXED2020.parquet` (71 sub-basins x 33 years = 2,343 records)

**Thesis reference:** Section 3.3.1 (methodology), Table 3.1 (SWAT files), Table 3.2 (output columns)


## 1. Configuration and imports


In [ ]:
import re
import os
import pandas as pd
import numpy as np
from pathlib import Path
from collections import Counter
from io import StringIO

# Repository paths - adjust these to match your local setup
REPO_ROOT = Path("..")

# SWAT raw input files (not included in repository; obtain from ICARDA)
SWAT_INPUT_DIR = REPO_ROOT / "TxtInOut"

# Output directory
OUTPUT_DIR = REPO_ROOT / "swat-prepared"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print("SWAT input dir:", SWAT_INPUT_DIR)
print("Output dir:", OUTPUT_DIR)


## 2. Parse the raw output.sub file

The ArcSWAT output.sub file has two formatting problems that standard CSV readers cannot handle:

- The calendar year and sub-basin area are sometimes glued into a single token
  (e.g., `1988.10173E+03` where 1988 is the year and 0.10173E+03 is the area in km2)
- Year indices (1-33) are sometimes used instead of calendar years (1988-2020)

The parser below handles both cases by splitting the glued tokens and converting
year indices back to calendar years using the simulation start year.


In [ ]:
import re
from collections import Counter
import pandas as pd
from io import StringIO

def split_year_area_token(token: str):
    """
    Handles two ArcSWAT weird patterns:

    A) '1988.10173E+03'  -> YEAR=1988, AREA='0.10173E+03'
    B) '33.0.10173E+03'  -> YEAR_INDEX=33, AREA='0.10173E+03'  (year count, not calendar)

    Returns dict: {'year': int, 'area_str': str}  OR  {'year_index': int, 'area_str': str}
    or None if no match.
    """
    token = token.strip()

    # A) Calendar year glued to area
    m = re.match(r'^((?:19|20)\d{2})(\.\d+(?:E[+-]\d+)?)$', token)
    if m:
        year = int(m.group(1))
        area_str = "0" + m.group(2)   # '.10173E+03' -> '0.10173E+03'
        return {"year": year, "area_str": area_str}

    # B) Year index glued: 33.0.10173E+03
    m2 = re.match(r'^(\d+)\.0\.(\d+(?:E[+-]\d+)?)$', token)
    if m2:
        year_index = int(m2.group(1))
        area_str = "0." + m2.group(2)  # '0.10173E+03'
        return {"year_index": year_index, "area_str": area_str}

    return None


def parse_output_sub_positional(path: str, base_year_for_index=1987):
    """
    base_year_for_index=1987 means:
      year_index 33 -> 1987 + 33 = 2020  (matches your series 1988 - 2020)
    """
    # 1) read lines
    with open(path, "r", errors="ignore") as f:
        lines = [ln.rstrip("\n") for ln in f]

    # 2) find header line
    header_idx = None
    for i, ln in enumerate(lines):
        if ("AREAkm2" in ln) and ("PRECIP" in ln) and ("SUB" in ln):
            header_idx = i
            break
    if header_idx is None:
        raise ValueError("Could not find header line (look for 'SUB ... AREAkm2 ... PRECIPmm').")

    # 3) collect candidate data lines after header
    data_lines = []
    for ln in lines[header_idx + 1:]:
        if not ln.strip():
            continue
        toks = re.split(r"\s+", ln.strip())
        # Typical table lines start with BIGSUB then integer id
        if len(toks) >= 10 and toks[0].isalpha() and toks[1].isdigit():
            data_lines.append(toks)

    if not data_lines:
        raise ValueError("No data lines detected after header.")

    # 4) keep only lines with mode token length
    mode_len = Counter(map(len, data_lines)).most_common(1)[0][0]
    data_lines = [t for t in data_lines if len(t) == mode_len]

    # 5) FIX rows by splitting glued YEAR+AREA token
    fixed_rows = []
    bad_rows = []

    for toks in data_lines:
        subtype = toks[0]
        sub_id  = toks[1]
        mon     = toks[2]       # in your case always 0 (annual)
        glued   = toks[3]       # problematic token

        split = split_year_area_token(glued)
        if split is None:
            bad_rows.append(toks)
            continue

        # Determine calendar YEAR
        if "year" in split:
            year = split["year"]
        else:
            # Convert year index -> calendar year (so 33 -> 2020)
            year = base_year_for_index + split["year_index"]

        area_str = split["area_str"]

        # New row: SUBTYPE SUB MON YEAR AREA + rest
        new = [subtype, sub_id, mon, str(year), area_str] + toks[4:]
        fixed_rows.append(new)

    if not fixed_rows:
        raise ValueError("All rows failed to parse.")

    # 6) Define columns (29 expected for your file after YEAR insertion)
    cols = [
        "SUBTYPE","SUB","MON","YEAR","AREAkm2",
        "PRECIPmm","SNOMELTmm","PETmm","ETmm","SWmm","PERCmm","SURQmm","GW_Qmm","WYLDmm",
        "SYLDt_ha","ORGNkg_ha","ORGPkg_ha","NSURQkg_ha","SOLPkg_ha","SEDPkg_ha",
        "LATQmm","LATNO3kg_ha","GWNO3kg_ha","CHOLAmic_L","CBODUmg_L","DOXQmg_L",
        "TNO3kg_ha","QTILEmm","TVAPkg_ha"
    ]

    # Truncate in case file has fewer columns in your build
    ncols = len(fixed_rows[0])
    cols = cols[:ncols]

    df = pd.DataFrame(fixed_rows, columns=cols)

    # 7) numeric conversion
    for c in df.columns:
        if c == "SUBTYPE":
            continue
        df[c] = pd.to_numeric(df[c], errors="coerce")

    # 8) strong typing
    for c in ["SUB","MON","YEAR"]:
        if c in df.columns:
            df[c] = df[c].astype("Int64")

    # 9) sanity checks
    print("✅ Parsed output.sub:", df.shape)
    print("SUB range:", int(df["SUB"].min()), "-", int(df["SUB"].max()), "| unique:", df["SUB"].nunique())
    print("YEAR range:", int(df["YEAR"].min()), "-", int(df["YEAR"].max()), "| unique:", df["YEAR"].nunique())
    print("MON unique:", df["MON"].dropna().unique()[:10])

    key = [c for c in ["PRECIPmm","ETmm","PETmm","SURQmm","GW_Qmm","WYLDmm","SYLDt_ha"] if c in df.columns]
    print("\nKey stats:")
    print(df[key].describe().loc[["count","mean","min","max"]])

    if bad_rows:
        print(f"\n⚠️ Still skipped rows: {len(bad_rows)}")
        print("Example skipped:", bad_rows[0][:8])
    else:
        print("\n✅ Skipped rows: 0")

    return df


# --- USAGE ---
path_sub = str(SWAT_INPUT_DIR / "output.sub")
df_sub = parse_output_sub_positional(path_sub, base_year_for_index=1987)

out_csv = str(OUTPUT_DIR / "output_sub_clean.csv")
df_sub.to_csv(out_csv, index=False)
print("Saved:", out_csv)


## 3. Quality control checks

Five checks are applied to the parsed dataset, matching Section 3.3.1 of the thesis:

1. Consistency of sub-basin identifiers (71 expected, range 1-71)
2. Separation of temporal and area attributes (year and area are distinct columns)
3. Absence of duplicated sub-basin-year records
4. Completeness across the 1988-2020 period (33 years x 71 sub-basins = 2,343 records)
5. Preservation of original SWAT output units


In [ ]:
# Quality control checks on the parsed dataset

print("=== Quality Control ===")
print()

# Check 1: Sub-basin identifiers
n_subs = df_sub["SUB"].nunique()
sub_range = (int(df_sub["SUB"].min()), int(df_sub["SUB"].max()))
print(f"1. Sub-basin identifiers: {n_subs} unique, range {sub_range[0]}-{sub_range[1]}")
assert n_subs == 71, f"Expected 71 sub-basins, got {n_subs}"

# Check 2: Year and area are separate columns
assert "YEAR" in df_sub.columns, "YEAR column missing"
assert "AREAkm2" in df_sub.columns, "AREAkm2 column missing"
print("2. Year and area columns: present and separated")

# Check 3: No duplicate sub-basin-year records
dupes = df_sub.groupby(["SUB", "YEAR"]).size()
n_dupes = (dupes > 1).sum()
print(f"3. Duplicate sub-basin-year records: {n_dupes}")
assert n_dupes == 0, f"Found {n_dupes} duplicate records"

# Check 4: Completeness
years = df_sub["YEAR"].unique()
year_range = (int(min(years)), int(max(years)))
n_records = len(df_sub)
expected = 71 * 33  # 71 sub-basins x 33 years
print(f"4. Records: {n_records} (expected {expected}), years {year_range[0]}-{year_range[1]}")

# Check 5: Units preserved (spot check ranges)
for col, expected_min, expected_max in [
    ("PRECIPmm", 0, 2000), ("SURQmm", 0, 1000), 
    ("SYLDt_ha", 0, 500), ("PERCmm", 0, 500)]:
    if col in df_sub.columns:
        vmin, vmax = df_sub[col].min(), df_sub[col].max()
        print(f"5. {col}: range {vmin:.1f} to {vmax:.1f} (plausible: {expected_min}-{expected_max})")

print()
print("All quality control checks passed.")


## 4. Export the prepared dataset

Save the quality-controlled dataset as parquet (for the scenario engine) and CSV (for inspection).
This is the file that notebook 06 (scenario engine) reads as its input.


In [ ]:
# Export as parquet and CSV
parquet_path = OUTPUT_DIR / "output_sub_FULL_FIXED2020.parquet"
csv_path = OUTPUT_DIR / "output_sub_FULL_FIXED2020.csv"

df_sub.to_parquet(parquet_path, index=False)
df_sub.to_csv(csv_path, index=False)

print(f"Saved: {parquet_path} ({parquet_path.stat().st_size / 1024:.1f} KB)")
print(f"Saved: {csv_path} ({csv_path.stat().st_size / 1024:.1f} KB)")
print(f"Shape: {df_sub.shape}")
print(f"Columns: {list(df_sub.columns)}")


## Summary

This notebook parsed the raw SWAT output.sub file from ICARDA's 2024 ArcSWAT model run
and produced a clean, quality-controlled annual dataset for 71 sub-basins over 33 years
(1988-2020). The dataset contains 2,343 records with the following columns:

- SUB: sub-basin identifier (1-71)
- YEAR: calendar year (1988-2020)
- AREAkm2: sub-basin area
- PRECIPmm, PETmm, ETmm: climate and evapotranspiration
- SURQmm: surface runoff (scenario engine input)
- SYLDt_ha: sediment yield (scenario engine input)
- PERCmm: percolation / groundwater recharge (scenario engine input)
- WYLDmm: water yield
- SWmm: soil water content

The output parquet file is the input to notebook 06 (scenario engine construction).

Thesis references: Section 3.3.1 (SWAT output preparation), Table 3.1, Table 3.2.
